# 양자컴퓨팅 부트캠프 Day 4: QubitSquare 플랫폼 아키텍처 및 결과 분석 핸즈온

본 노트북은 **12시간 양자컴퓨팅 특강의 4일차 3교시 실습 교재**입니다.
현대적인 웹 기반 양자 플랫폼인 **QubitSquare**의 데이터 계층 구조(`Lab -> Design -> Circuit -> Run -> Result -> Compare`), 회로 자동 전체 측정 규칙, 엔디안 변환, 다중 결과 비교 분석(Compare)을 파이썬 코드로 모델링하여 실습합니다.

---

### 오늘 실습의 5대 학습 목표
1. **QubitSquare 데이터 계층**: Lab, Design, Circuit, Run, Result, Compare의 독립성과 관계 이해.
2. **회로 자동 측정 규칙**: Measure 게이트 미배치 시 전체 큐비트 자동 측정 삽입 vs 지정 큐비트 측정 메커니즘.
3. **엔디안(Endianness) 변환**: Qiskit Little-Endian($|q_1 q_0\rangle$)과 QubitSquare 와이어 배치 순서 간의 매핑.
4. **Compare 다중 결과 분석**: 2~8개의 Result를 선택하여 샷 수 차이 및 엔진 간 오차를 비교하는 알고리즘 구현.
5. **실행 환경의 이해**: 브라우저 로컬 시뮬레이터(WASM)와 원격 연산 엔진의 실질적 특성 파악.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field

print('QubitSquare 아키텍처 모델링 준비 완료.')


## 1. QubitSquare 핵심 데이터 모델

- **Lab**: 연구실 단위의 최상위 프로젝트 공간
- **Design**: 회로, 문제 정의, Score, Optimizer, Display 설정을 묶는 명세 단위
- **Circuit**: 독립적으로 조립되는 양자 회로 와이어 및 게이트 배열
- **Run**: **Design 1개 $\times$ Compute Engine 1개**의 독립 실행 작업
- **Result**: 특정 Run에서 획득한 측정 bitstring counts 분포 스냅샷
- **Compare**: 성공한 2~8개의 Result를 대조하는 비교 분석 뷰


In [ ]:
@dataclass(frozen=True)
class QubitSquareGate:
    name: str
    qubits: list[int]
    params: list[float] = field(default_factory=list)

@dataclass
class QubitSquareCircuit:
    num_qubits: int
    gates: list[QubitSquareGate] = field(default_factory=list)

    def add_gate(self, name: str, qubits: list[int], params: list[float] | None = None) -> None:
        self.gates.append(QubitSquareGate(name, qubits, params or []))

    def resolve_measurements(self) -> list[int]:
        explicit = [g.qubits[0] for g in self.gates if g.name.lower() == 'measure']
        if not explicit:
            return list(range(self.num_qubits))
        return sorted(set(explicit))

@dataclass
class QubitSquareRunResult:
    run_id: str
    design_name: str
    engine: str
    shots: int
    profile: str
    counts: dict


## 2. 자동 전체 측정 규칙 테스트

사용자가 회로에 `Measure` 게이트를 수동으로 배치했는지 여부에 따른 실행 엔진의 자동 측정 규칙을 검증합니다.


In [ ]:
# 1) Measure 게이트 미배치 회로 (H, CX만 존재)
circ_auto = QubitSquareCircuit(num_qubits=2)
circ_auto.add_gate('H', [0])
circ_auto.add_gate('CX', [0, 1])

print('1) Measure 미배치 회로:')
print('   자동 삽입 측정 대상 큐비트:', circ_auto.resolve_measurements(), '(0, 1 전체 자동 측정)')

# 2) q0에만 명시적으로 Measure 배치한 회로
circ_explicit = QubitSquareCircuit(num_qubits=2)
circ_explicit.add_gate('H', [0])
circ_explicit.add_gate('CX', [0, 1])
circ_explicit.add_gate('Measure', [0])

print('\n2) q0에만 Measure 배치한 회로:')
print('   실제 측정 대상 큐비트:', circ_explicit.resolve_measurements(), '(지정된 q0만 측정)')


## 3. 비트 순서 변환 (Qiskit vs QubitSquare)

- **Qiskit (Little-Endian)**: 맨 오른쪽이 0번 큐비트 ($|q_{n-1} \dots q_0\rangle$)
- **QubitSquare 화면 배치**: 화면 상단 와이어부터 $q_0, q_1, \dots$ 순서


In [ ]:
def convert_qiskit_to_qubitsquare_bitstring(qiskit_bitstring: str) -> str:
    return qiskit_bitstring[::-1]

qiskit_val = '10'  # q1=1, q0=0
qs_val = convert_qiskit_to_qubitsquare_bitstring(qiskit_val)
print(f'Qiskit 표기 "{qiskit_val}" (q1=1, q0=0) -> QubitSquare 와이어 순서: "{qs_val}"')


## 4. Compare 다중 결과 비교 분석 알고리즘

성공한 2~8개의 Result를 선택해 확률 분포 테이블을 생성하는 Compare 로직을 구현합니다.
Shot 수가 다른 경우(예: 100 shots vs 1000 shots) 측정 비율을 정규화하여 대조합니다.


In [ ]:
def create_qubitsquare_compare(results: list[QubitSquareRunResult]) -> dict[str, dict[str, float]]:
    if not (2 <= len(results) <= 8):
        raise ValueError('Compare는 2개에서 8개 사이의 Result만 선택할 수 있습니다.')

    all_keys = sorted(set(k for r in results for k in r.counts.keys()))
    table = {k: {} for k in all_keys}

    for r in results:
        col_label = f'{r.design_name} [{r.engine}, {r.shots}s]'
        for k in all_keys:
            count = r.counts.get(k, 0)
            prob = count / r.shots
            table[k][col_label] = round(prob, 4)

    return table

res_100 = QubitSquareRunResult(
    run_id='run_01', design_name='Bell_Experiment', engine='Q-Vista WASM',
    shots=100, profile='browser_wasm', counts={'00': 46, '11': 54}
)

res_1000 = QubitSquareRunResult(
    run_id='run_02', design_name='Bell_Experiment', engine='Q-Vista WASM',
    shots=1000, profile='browser_wasm', counts={'00': 498, '11': 502}
)

compare_table = create_qubitsquare_compare([res_100, res_1000])

print('[Compare 다중 결과 비교 테이블]')
print(f"{'State':<8} | {'Bell (100 shots)':<25} | {'Bell (1000 shots)':<25}")
print('-' * 65)
for state, cols in compare_table.items():
    vals = list(cols.values())
    print(f"{state:<8} | {vals[0]:<25.4f} | {vals[1]:<25.4f}")


---
### 12시간 특강 마무리 및 안내
- **Day 1**: 단일 큐비트 상태벡터와 Born 측정 규칙의 선형대수학적 기초
- **Day 2**: 텐서곱과 다중 큐비트 얽힘(Bell 상태)의 비분리성
- **Day 3**: 양자 간섭을 통한 진폭 증폭과 Grover 탐색 알고리즘
- **Day 4**: QubitSquare 웹 플랫폼을 통한 실무적 회로 설계와 산업 문제 분석

수고 많으셨습니다!
